
# JABARULIN AI  
## Hybrid NLP-Based Tourism Recommendation System

Sistem rekomendasi wisata Jawa Barat berbasis AI menggunakan:
- IndoBERT → Intent Classification
- TF-IDF → Recommendation Retrieval
- Cosine Similarity → Similarity Search
- Weighted Score → Ranking Recommendation


##1. Setup Environment & Library Imports

In [1]:
!pip install transformers datasets torch scikit-learn

import pandas as pd
import numpy as np
import torch
import pickle
import os
import random

import shutil
from google.colab import files

# Hugging Face Transformers
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
    set_seed
)

# Scikit-Learn untuk ML Tradisional & Evaluasi
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, ndcg_score
from sklearn.model_selection import train_test_split

# PyTorch untuk Dataset Management
from torch.utils.data import Dataset

# Mengunci random seed agar hasil selalu sama
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
set_seed(SEED)

##2. Data Loading & Preprocessing

In [2]:
print("1. Membaca Dataset Final Jabarulin...")
df = pd.read_csv('dataset_final_jabarulin.csv')

# Mengambil kolom yang relevan dan normalisasi nama
tourism_df = df[['name', 'category', 'intent_label', 'rating', 'cleaned_reviews']].copy()
tourism_df['name'] = tourism_df['name'].astype(str).str.title().str.strip()

# Handle Missing Value (Mencegah Error saat komputasi)
tourism_df['rating'] = tourism_df['rating'].fillna(tourism_df['rating'].mean())
tourism_df['cleaned_reviews'] = tourism_df['cleaned_reviews'].fillna('')

print(f"Jumlah Data: {len(tourism_df)} baris")

print("\n2. Encoding Label Intent...")
le_intent = LabelEncoder()
tourism_df['intent_encoded'] = le_intent.fit_transform(tourism_df['intent_label'])

# Menampilkan hasil mapping (Misal: 0 = adventure, 1 = camping, dll)
print(f"Mapping Label Intent: {dict(zip(le_intent.classes_, le_intent.transform(le_intent.classes_)))}")

1. Membaca Dataset Final Jabarulin...
Jumlah Data: 358 baris

2. Encoding Label Intent...
Mapping Label Intent: {'adventure': np.int64(0), 'camping': np.int64(1), 'gunung': np.int64(2), 'healing': np.int64(3), 'keluarga': np.int64(4), 'pantai': np.int64(5)}


##3. Preparation for IndoBERT Fine-Tuning

In [3]:
print("1. Inisialisasi IndoBERT Tokenizer & Model...")
model_name = "indobenchmark/indobert-base-p1"
tokenizer = AutoTokenizer.from_pretrained(model_name)

num_intent_labels = len(le_intent.classes_)
model_intent = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_intent_labels
)

print("\n2. Membagi Dataset (Train 80% & Val 20%) dengan Stratify...")
train_df, val_df = train_test_split(
    tourism_df,
    test_size=0.2,
    stratify=tourism_df['intent_encoded'],
    random_state=42
)
print(f"Data Latih: {len(train_df)} baris | Data Validasi: {len(val_df)} baris")

print("\n3. Setup Custom PyTorch Dataset...")
class JabarulinDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=64):
        self.texts = texts.reset_index(drop=True)
        self.labels = labels.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]

        # Tokenisasi tanpa padding statis (Dinamis via DataCollator)
        encoding = self.tokenizer(
            text, truncation=True, max_length=self.max_len
        )

        item = {key: torch.tensor(val) for key, val in encoding.items()}
        item['labels'] = torch.tensor(label, dtype=torch.long)
        return item

train_dataset = JabarulinDataset(train_df['cleaned_reviews'], train_df['intent_encoded'], tokenizer)
val_dataset = JabarulinDataset(val_df['cleaned_reviews'], val_df['intent_encoded'], tokenizer)

# Menyiapkan Data Collator untuk efisiensi memori (Dynamic Padding)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

1. Inisialisasi IndoBERT Tokenizer & Model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/1.53k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/229k [00:00<?, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/498M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/498M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: indobenchmark/indobert-base-p1
Key               | Status  | 
------------------+---------+-
classifier.bias   | MISSING | 
classifier.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



2. Membagi Dataset (Train 80% & Val 20%) dengan Stratify...
Data Latih: 286 baris | Data Validasi: 72 baris

3. Setup Custom PyTorch Dataset...


##4. Training Execution & Model Export

In [4]:
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    return {'accuracy': accuracy_score(labels, preds)}

print("1. Konfigurasi Parameter Training...")
training_args = TrainingArguments(
    output_dir='./results_indobert_intent',
    num_train_epochs=5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy="epoch",
    save_strategy="epoch"
)

trainer = Trainer(
    model=model_intent,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

print("\n🚀 2. MEMULAI FINE-TUNING INDOBERT...")
trainer.train()

print("\n💾 3. MENYIMPAN MODEL UNTUK BACKEND...")
# Menyimpan seluruh otak AI ke dalam folder lokal
save_path = './jabarulin_indobert_model'
os.makedirs(save_path, exist_ok=True)

trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)

# Menyimpan Label Encoder agar Backend tahu arti angka prediksinya
with open(f'{save_path}/label_encoder.pkl', 'wb') as f:
    pickle.dump(le_intent, f)

print(f"Model dan Tokenizer berhasil disimpan di folder '{save_path}'!")

1. Konfigurasi Parameter Training...

🚀 2. MEMULAI FINE-TUNING INDOBERT...


Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.833064,0.652778
2,No log,0.429197,0.875000
3,No log,0.297987,0.916667
4,No log,0.354785,0.916667
5,No log,0.329484,0.888889


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


💾 3. MENYIMPAN MODEL UNTUK BACKEND...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model dan Tokenizer berhasil disimpan di folder './jabarulin_indobert_model'!


##5. Global TF-IDF Fallback Setup

In [5]:
print("Membuat Global TF-IDF Matrix untuk sistem Fallback...")

tfidf_global = TfidfVectorizer(
    stop_words=None,
    ngram_range=(1,2)
)

tfidf_matrix_global = tfidf_global.fit_transform(tourism_df['cleaned_reviews'])

print('TF-IDF Global Matrix Shape:', tfidf_matrix_global.shape)

Membuat Global TF-IDF Matrix untuk sistem Fallback...
TF-IDF Global Matrix Shape: (358, 196567)


##6. Core AI Functions (Hybrid Architecture)

In [6]:
# 1. NLP Semantic Understanding (IndoBERT Inference)
def predict_intent_indobert(user_input):
    model_intent.eval()
    inputs = tokenizer(
        user_input, return_tensors="pt", truncation=True, padding=True, max_length=64
    ).to(model_intent.device)

    with torch.no_grad():
        outputs = model_intent(**inputs)

    predicted_class_id = outputs.logits.argmax(dim=-1).item()
    return le_intent.inverse_transform([predicted_class_id])[0]


# 2. Global Recommendation (Jaring Pengaman / Fallback)
def global_recommendation(user_input, top_n=5, similarity_threshold=0.03):
    user_vector = tfidf_global.transform([user_input])
    similarity_scores = cosine_similarity(user_vector, tfidf_matrix_global).flatten()

    global_df = tourism_df.copy()
    global_df['similarity_score'] = similarity_scores
    global_df = global_df[global_df['similarity_score'] > similarity_threshold].copy()

    if global_df.empty:
        return pd.DataFrame()

    global_df['final_score'] = (global_df['similarity_score'] * 0.7) + ((global_df['rating'] / 5) * 0.3)
    global_df = global_df.sort_values(by='final_score', ascending=False).drop_duplicates(subset='name')

    recommendations = global_df.head(top_n).copy()
    recommendations[['similarity_score', 'final_score']] = recommendations[['similarity_score', 'final_score']].round(3)
    recommendations['rating'] = recommendations['rating'].round(1)

    return recommendations


# 3. Final Hybrid Recommendation (IndoBERT + TF-IDF)
def hybrid_recommendation(user_input, top_n=5, similarity_threshold=0.05):
    # Layer 1: NLP Intent Classification
    predicted_intent = predict_intent_indobert(user_input)
    print(f'>> Predicted Intent (IndoBERT): {predicted_intent.upper()}')

    # Layer 2: Dataset Filtering
    filtered_df = tourism_df[tourism_df['intent_label'] == predicted_intent].copy()

    # Layer 3: Similarity Search
    filtered_tfidf = TfidfVectorizer(stop_words=None, ngram_range=(1,2))
    filtered_matrix = filtered_tfidf.fit_transform(filtered_df['cleaned_reviews'])
    user_vector = filtered_tfidf.transform([user_input])

    similarity_scores = cosine_similarity(user_vector, filtered_matrix).flatten()
    filtered_df['similarity_score'] = similarity_scores
    filtered_df = filtered_df[filtered_df['similarity_score'] > similarity_threshold].copy()

    # Layer 4: Fallback Handling
    if len(filtered_df) < top_n:
        print('>> Peringatan: Data kurang, melakukan Fallback ke Global Recommendation...\n')
        return global_recommendation(user_input, top_n, similarity_threshold=0.03)

    # Layer 5: Ranking & Weighted Score (70% Teks, 30% Rating Google Maps)
    filtered_df['final_score'] = (filtered_df['similarity_score'] * 0.7) + ((filtered_df['rating'] / 5) * 0.3)
    filtered_df = filtered_df.sort_values(by='final_score', ascending=False).drop_duplicates(subset='name')

    recommendations = filtered_df.head(top_n).copy()
    recommendations[['similarity_score', 'final_score']] = recommendations[['similarity_score', 'final_score']].round(3)
    recommendations['rating'] = recommendations['rating'].round(1)

    return recommendations

##7. System Testing (Inference)

In [7]:
queries = [
    'pantai sunset bagus dan sepi',
    'tempat healing dingin dekat bandung',
    'rafting dan offroad seru',
    'wisata keluarga untuk anak anak',
    'tempat wisata romantis sunset'
]

for query in queries:
    print('=' * 80)
    print(f'USER INPUT: "{query}"')
    print('=' * 80)

    recommendations = hybrid_recommendation(query, top_n=3)

    if not recommendations.empty:
        print(recommendations[['name', 'intent_label', 'category', 'rating', 'final_score']])
    else:
        print("Tidak ada rekomendasi yang ditemukan.")
    print("\n")

USER INPUT: "pantai sunset bagus dan sepi"
>> Predicted Intent (IndoBERT): ADVENTURE
                            name intent_label  category  rating  final_score
340      Stone Garden Padalarang    adventure  trekking     4.5        0.465
148  Rafting Pangalengan Bandung    adventure   rafting     5.0        0.345
59         Situ Cileunca Rafting    adventure   rafting     4.5        0.314


USER INPUT: "tempat healing dingin dekat bandung"
>> Predicted Intent (IndoBERT): GUNUNG
                 name intent_label       category  rating  final_score
347  Tangkuban Perahu       gunung  gunung berapi     4.5        0.380
354    Gunung Puntang       gunung  puncak gunung     4.5        0.325
216   Gunung Koromong       gunung  puncak gunung     4.4        0.312


USER INPUT: "rafting dan offroad seru"
>> Predicted Intent (IndoBERT): ADVENTURE
                                     name intent_label category  rating  \
333                        Cikole Offroad    adventure  offroad     4.5   

##8. Academic Evaluation (NDCG Score)

In [8]:
print("=== MENGHITUNG NILAI NDCG SCORE ===")
query_eval = 'tempat healing dingin dekat bandung'
eval_result = hybrid_recommendation(query_eval, top_n=5)

recommended_scores = eval_result['final_score'].tolist()

if len(recommended_scores) < 2:
    print('NDCG tidak bisa dihitung karena rekomendasi kurang dari 2 item.')
else:
    predicted_scores = np.asarray([recommended_scores])
    relevance_length = len(recommended_scores)
    # Asumsi: Item peringkat pertama paling relevan, menurun ke bawah
    true_relevance = np.asarray([list(range(relevance_length, 0, -1))])

    ndcg = ndcg_score(true_relevance, predicted_scores)
    print(f'\nNDCG Score untuk query "{query_eval}": {ndcg:.4f}')
    print("(Score 1.0000 menandakan bahwa sistem berhasil mengurutkan tempat wisata dengan relevansi yang sempurna)")

=== MENGHITUNG NILAI NDCG SCORE ===
>> Predicted Intent (IndoBERT): GUNUNG

NDCG Score untuk query "tempat healing dingin dekat bandung": 1.0000
(Score 1.0000 menandakan bahwa sistem berhasil mengurutkan tempat wisata dengan relevansi yang sempurna)


## 9. EXPORT & AUTO-DOWNLOAD ARTIFACTS

In [9]:
print("1. Membuat file requirements.txt menggunakan pip freeze...")
# Menjalankan perintah shell pip freeze dan menyimpannya ke requirements.txt
!pip freeze > requirements.txt
print("requirements.txt berhasil dibuat beserta versi library-nya")

print("\n2. Mengompres folder model menjadi ZIP")
folder_to_zip = './jabarulin_indobert_model'
zip_filename = 'jabarulin_indobert_model'

shutil.make_archive(zip_filename, 'zip', folder_to_zip)
print(f"Folder model berhasil di-ZIP menjadi {zip_filename}.zip!")

print("\n3. Memulai proses download otomatis ke laptop")
files.download(f"{zip_filename}.zip")
files.download('requirements.txt')

1. Membuat file requirements.txt menggunakan pip freeze...
requirements.txt berhasil dibuat beserta versi library-nya

2. Mengompres folder model menjadi ZIP
Folder model berhasil di-ZIP menjadi jabarulin_indobert_model.zip!

3. Memulai proses download otomatis ke laptop


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>